## 📦 STEP 1: Installation

In [ ]:
print("📦 Installing dependencies...\n")

!pip install -q transformers==4.43.3 datasets accelerate peft bitsandbytes sentencepiece sacrebleu torch

print("Installing IndicTransToolkit...")
!git clone https://github.com/VarunGumma/IndicTransToolkit 2>/dev/null || echo "Already cloned"
!cd IndicTransToolkit && pip install -q --editable ./

print("\n✅ Installation complete!")
print("⚠️ RESTART RUNTIME: Runtime → Restart runtime")
print("   Then continue from STEP 2")

---

# ⚠️ RESTART RUNTIME NOW ⚠️

---

## 🔧 STEP 2: Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")

:## 🔐 STEP 3: HuggingFace Login

In [ ]:
from huggingface_hub import notebook_login
print("🔐 Login to HuggingFace")
print("   Get token from: https://huggingface.co/settings/tokens")
notebook_login()

## ⚙️ STEP 4: Configuration - 150K SAMPLES

In [ ]:
import torch
import os

# GPU Check
print(f"🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Paths
WORKSPACE = "/content/drive/MyDrive/bhasha_workspace"
CACHE_DIR = f"{WORKSPACE}/preprocessed_cache"  # Store preprocessed data here
os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Model
BASE_MODEL = "ai4bharat/indictrans2-en-indic-dist-200M"
SRC_LANG = "eng_Latn"
TGT_LANG = "mar_Deva"

# Dataset
DATASET_NAME = "Hemanth-thunder/english-to-marathi-mt"
SRC_COL = "en"
TGT_COL = "mr"

# Training config - SCALED FOR 150K
TRAIN_SIZE = 150000      # 1.5 lakh samples
VAL_SIZE = 5000
TEST_SIZE = 5000
NUM_EPOCHS = 2           # 2 epochs for better quality
BATCH_SIZE = 8           # Reduced to fit memory
GRAD_ACCUM = 4           # Effective batch = 8 * 4 = 32
LEARNING_RATE = 3e-5     # Slightly lower for more data
MAX_LEN = 128

# LoRA config
LORA_R = 16              # Increased rank for more capacity
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

OUTPUT_DIR = f"{WORKSPACE}/models/indictrans2_lora_150k"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\n✅ Configuration for 150K samples:")
print(f"   Model: {BASE_MODEL}")
print(f"   Train samples: {TRAIN_SIZE:,}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Gradient accumulation: {GRAD_ACCUM}")
print(f"   Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"   LoRA rank: {LORA_R}")
print(f"\n📊 Training stats:")
print(f"   Steps per epoch: {TRAIN_SIZE // (BATCH_SIZE * GRAD_ACCUM):,}")
print(f"   Total steps: {(TRAIN_SIZE // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS:,}")
print(f"   Expected time: 3-4 hours")
print(f"   Expected BLEU: 36-40")

## 📥 STEP 5: Load & Clean Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import gc

print("📥 Loading dataset...")
dataset = load_dataset(DATASET_NAME)
df_full = dataset['train'].to_pandas()

print(f"Raw dataset size: {len(df_full):,}")

# Clean
df_full = df_full.dropna(subset=[SRC_COL, TGT_COL])
df_full = df_full.drop_duplicates()
df_full = df_full[(df_full[SRC_COL].str.strip() != '') & (df_full[TGT_COL].str.strip() != '')]
df_full = df_full[(df_full[SRC_COL].str.len() < 500) & (df_full[TGT_COL].str.len() < 500)]

print(f"After cleaning: {len(df_full):,}")

# Sample
TOTAL = TRAIN_SIZE + VAL_SIZE + TEST_SIZE
if len(df_full) >= TOTAL:
    df_subset = df_full.sample(n=TOTAL, random_state=42).reset_index(drop=True)
else:
    print(f"⚠️ Warning: Only {len(df_full):,} samples available, using all")
    df_subset = df_full.reset_index(drop=True)
    TRAIN_SIZE = int(len(df_subset) * 0.9)
    VAL_SIZE = int(len(df_subset) * 0.05)
    TEST_SIZE = len(df_subset) - TRAIN_SIZE - VAL_SIZE

del df_full, dataset
gc.collect()

# Split
train_df, temp_df = train_test_split(df_subset, train_size=TRAIN_SIZE, random_state=42)
val_df, test_df = train_test_split(temp_df, train_size=VAL_SIZE, test_size=TEST_SIZE, random_state=42)
del df_subset, temp_df
gc.collect()

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"\n✅ Final splits: {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")

## 🤖 STEP 6: Load Model with LoRA

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

print("⚙️ Loading distilled 200M model...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    use_auth_token=True
)

# Load base model
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    use_auth_token=True
).to(DEVICE)

print(f"✅ Base model loaded: {sum(p.numel() for p in base_model.parameters()) / 1e6:.1f}M params")
print(f"   GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# LoRA config - RANK 16 for more capacity
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(base_model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"\n✅ LoRA applied (r={LORA_R}):")
print(f"   Trainable: {trainable / 1e6:.2f}M ({100 * trainable / total:.2f}%)")
print(f"   Total: {total / 1e6:.1f}M")
print(f"   GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# ❌ REMOVED: ip = IndicProcessor(inference=True)
print(f"\n✅ Using direct tokenization (no IndicProcessor)")
print(f"   CPU RAM saved: ~850 MB")

## 🔥 STEP 7: Pre-process & SAVE TO DISK (Critical for 150K!)

**For 150K samples, we MUST save preprocessed data to disk to avoid RAM issues.**

In [ ]:
from torch.utils.data import Dataset
import torch

class IndicTrans2Dataset(Dataset):
    """
    Dataset for IndicTrans2 - follows required format
    Format: "src_lang tgt_lang actual_text"
    """
    def __init__(self, df, tokenizer, src_col, tgt_col, src_lang, tgt_lang, max_len=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.src_col = src_col
        self.tgt_col = tgt_col
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Get text and clean
        src_text = str(row[self.src_col]).strip()
        tgt_text = str(row[self.tgt_col]).strip()

        # ✅ CRITICAL: Format MUST be "src_lang tgt_lang text"
        src_input = f"{self.src_lang} {self.tgt_lang} {src_text}"
        tgt_input = f"{self.tgt_lang} {self.src_lang} {tgt_text}"

        # Tokenize source
        inputs = self.tokenizer(
            src_input,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Tokenize target
        with self.tokenizer.as_target_tokenizer():
            targets = self.tokenizer(
                tgt_input,
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )

        # Create labels
        labels = targets["input_ids"].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'labels': labels.squeeze(0)
        }


print("="*70)
print("🚀 CREATING DATASETS - INDICTRANS2 FORMAT")
print("="*70)
print("Using correct format: 'src_lang tgt_lang text'\n")

train_dataset = IndicTrans2Dataset(
    train_df, tokenizer, SRC_COL, TGT_COL,
    SRC_LANG, TGT_LANG, MAX_LEN
)

val_dataset = IndicTrans2Dataset(
    val_df, tokenizer, SRC_COL, TGT_COL,
    SRC_LANG, TGT_LANG, MAX_LEN
)

print(f"✅ Datasets ready:")
print(f"   Train: {len(train_dataset):,} samples")
print(f"   Val: {len(val_dataset):,} samples")
print(f"   Format: {SRC_LANG} {TGT_LANG} <text>")
print(f"\n⚡ Ready to train!")

## 🚀 STEP 9: Training (3-4 hours)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import time

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# Training arguments - OPTIMIZED FOR 150K
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # Optimization
    learning_rate=LEARNING_RATE,
    warmup_steps=500,
    weight_decay=0.01,
    optim="paged_adamw_8bit",

    # Evaluation
    eval_strategy="steps",
    eval_steps=2000,

    # Saving
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,

    # Performance
    fp16=False,
    bf16=True,
    gradient_checkpointing=False,
    dataloader_num_workers=0,  # ✅ SET TO 0
    dataloader_pin_memory=True,

    # Logging
    logging_steps=100,
    report_to=[],

    # Misc
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("="*70)
print("🚀 STARTING TRAINING - 150K SAMPLES")
print("="*70)
print(f"\nConfiguration:")
print(f"  Samples: {len(train_dataset):,}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"  Steps per epoch: {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM):,}")
print(f"  Total steps: {(len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS:,}")
print(f"\nExpected:")
print(f"  Speed: 0.8-1.2 it/s")
print(f"  Time: 3-4 hours")
print(f"  Final BLEU: 36-40")
print(f"\nStarting...\n")

start_time = time.time()
train_result = trainer.train()
elapsed = (time.time() - start_time) / 3600

print("\n" + "="*70)
print("✅ TRAINING COMPLETE")
print("="*70)
print(f"\nTime: {elapsed:.2f} hours")
print(f"Loss: {train_result.training_loss:.4f}")
print(f"Steps: {train_result.global_step}")

## 📊 STEP 10: Evaluation

In [ ]:
import sacrebleu
import torch
import time
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, util

# Load once (outside function ideally)
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

def evaluate_full(model, tokenizer, test_df, src_lang, tgt_lang, n_samples=1000):
    """
    Full evaluation:
    - BLEU
    - Semantic Similarity
    - Words Per Minute (WPM)
    """

    model.eval()

    sample_df = test_df.sample(n=min(n_samples, len(test_df)), random_state=42)
    sources = sample_df[SRC_COL].tolist()
    references = sample_df[TGT_COL].tolist()

    predictions = []
    total_words = 0

    batch_size = 8
    start_time = time.time()

    for i in tqdm(range(0, len(sources), batch_size), desc="Evaluating"):
        batch = sources[i:i+batch_size]

        # IndicTrans2 format
        batch_formatted = [f"{src_lang} {tgt_lang} {text}" for text in batch]

        inputs = tokenizer(
            batch_formatted,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=MAX_LEN,
                num_beams=5,
                early_stopping=True
            )

        with tokenizer.as_target_tokenizer():
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # Clean outputs
        cleaned = []
        for text in decoded:
            parts = text.split(" ", 2)
            if len(parts) >= 3 and parts[0] in [src_lang, tgt_lang] and parts[1] in [src_lang, tgt_lang]:
                final_text = parts[2]
            else:
                final_text = text

            cleaned.append(final_text)

            # Count words
            total_words += len(final_text.split())

        predictions.extend(cleaned)

    end_time = time.time()
    total_time_sec = end_time - start_time

    # ======================
    # 🔹 BLEU
    # ======================
    refs_formatted = [[ref] for ref in references]
    bleu = sacrebleu.corpus_bleu(predictions, refs_formatted)

    # ======================
    # 🔹 Semantic Similarity
    # ======================
    emb_preds = semantic_model.encode(predictions, convert_to_tensor=True, batch_size=64)
    emb_refs = semantic_model.encode(references, convert_to_tensor=True, batch_size=64)

    cosine_scores = util.cos_sim(emb_preds, emb_refs).diagonal()
    semantic_score = cosine_scores.mean().item()

    # ======================
    # 🔹 Words Per Minute
    # ======================
    wpm = (total_words / total_time_sec) * 60

    return {
        "bleu": bleu.score,
        "semantic_similarity": semantic_score,
        "wpm": wpm,
        "predictions": predictions,
        "references": references,
        "sample_df": sample_df
    }


# ======================
# 🚀 RUN
# ======================

print("📊 Evaluating on 1000 samples...\n")

results = evaluate_full(
    trainer.model, tokenizer, test_df, SRC_LANG, TGT_LANG, n_samples=1000
)

bleu = results["bleu"]
semantic = results["semantic_similarity"]
wpm = results["wpm"]
preds = results["predictions"]
refs = results["references"]
sample_df = results["sample_df"]

print(f"\n{'='*70}")
print(f"BLEU SCORE           : {bleu:.2f}")
print(f"SEMANTIC SIMILARITY  : {semantic:.4f}")
print(f"WORDS PER MINUTE     : {wpm:.2f}")
print(f"{'='*70}")

# Quality Interpretation
if bleu >= 36:
    print("\n🎉 EXCELLENT!")
elif bleu >= 32:
    print("\n✅ GREAT!")
elif bleu >= 28:
    print("\n✅ GOOD!")
else:
    print("\n⚠️ Consider training 1 more epoch")

print(f"\n📝 Sample Translations:\n")
for i in range(5):
    print(f"{i+1}. EN:  {sample_df.iloc[i][SRC_COL][:70]}")
    print(f"   REF: {refs[i][:70]}")
    print(f"   OUT: {preds[i][:70]}\n")